# 04 · Model Training

Fine-tunes an ImageNet-pretrained ResNet-18 on the labelled frame dataset.

Two variants are trained from the same relabelled dataset (Section III-E of
the report) to isolate the effect of sampler strength on class imbalance:

- **v2** — full inverse-frequency `WeightedRandomSampler` weighting
- **v3** — softened weighting (`w^0.3`) to avoid overfitting the rarest
  (`left`, 168-frame) class

v3 scores higher on held-out test accuracy (80.5% vs 77.7%), but v2 has
better recall on the rare `right` class and was the one that actually
worked when deployed on the track — see the README and report for why
test accuracy was a poor predictor of deployment performance here.

## Setup (Colab only — skip locally)

In [ ]:
# Only needed if running on Google Colab with data stored on Drive.
# Skip this cell entirely for a local run.
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks')

In [ ]:
# unzip frames
import os
import zipfile

if not os.path.exists('Frames'):
    print('Unzipping Frames.zip...')
    with zipfile.ZipFile('Frames.zip', 'r') as z:
        z.extractall('.')
    print('Done')
else:
    print('Frames folder already exists, skipping unzip')

## Video-level train / val / test split

Splitting by *video*, not by frame, stops near-identical adjacent frames leaking across the split — see Section III-D of the report.

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

df = pd.read_csv('labels.xls')

df['video'] = df['image'].apply(lambda p: p.replace('\\', '/').split('/')[1])

def get_direction(video_name):
    base = os.path.splitext(video_name)[0]
    return 'direction_b' if '_l' in base else 'direction_a'

df['direction'] = df['video'].apply(get_direction)

print('Total frames:', len(df))
print('\nFrames per direction:')
print(df['direction'].value_counts())
print('\nLabel distribution:')
print(df['label'].value_counts())

In [ ]:
# split by video, not by frame - 4 train, 1 val, 1 test per direction
fwd_vids = sorted(df[df['direction'] == 'direction_a']['video'].unique())
bwd_vids = sorted(df[df['direction'] == 'direction_b']['video'].unique())

def split_6(vids, seed=42):
    train, temp = train_test_split(vids, test_size=2, random_state=seed)
    val, test = train_test_split(temp, test_size=1, random_state=seed)
    return list(train), [val[0]], [test[0]]

fwd_train, fwd_val, fwd_test = split_6(fwd_vids)
bwd_train, bwd_val, bwd_test = split_6(bwd_vids)

all_train = fwd_train + bwd_train
all_val = fwd_val + bwd_val
all_test = fwd_test + bwd_test

train_df = df[df['video'].isin(all_train)].reset_index(drop=True)
val_df = df[df['video'].isin(all_val)].reset_index(drop=True)
test_df = df[df['video'].isin(all_test)].reset_index(drop=True)

print(f'Train: {len(train_df)} frames')
print(f'Val:   {len(val_df)} frames')
print(f'Test:  {len(test_df)} frames')

train_df.to_csv('train.xls', index=False)
val_df.to_csv('val.xls', index=False)
test_df.to_csv('test.xls', index=False)

## (Colab only) copy frames to local disk

Speeds up dataloading vs. reading from a mounted Drive. Skip locally.

In [ ]:
import shutil

print('Copying Frames to local disk...')
shutil.copytree('Frames', '/content/Frames')

for csv_file in ['train.xls', 'val.xls', 'test.xls']:
    csv_df = pd.read_csv(csv_file)
    csv_df['image'] = csv_df['image'].apply(lambda p: '/content/' + p.replace('\\', '/'))
    csv_df.to_csv(csv_file, index=False)
    print(f'Updated {csv_file}')

print('Done')

## Dataset, transforms, and class-imbalance handling

The 24:1 ratio between the most and least represented classes is handled two
ways (Section IV-C): class-weighted cross-entropy loss, and a
`WeightedRandomSampler` on the training split. `make_loaders()` below takes
a `soften_exponent` — `1.0` for full inverse-frequency weighting (v2),
`0.3` for the softened version that avoids overfitting the rarest class (v3).

In [ ]:
import time
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch import nn, optim
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

CLASS_NAMES = ['hard_left', 'left', 'straight', 'right', 'hard_right']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training on:', device)


class RobotDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image'].replace('\\', '/')  # Windows -> Linux/Colab paths
        img = Image.open(img_path).convert('RGB')
        w, h = img.size
        img = img.crop((0, 200, w, h))  # match preprocessing used at inference
        if self.transform:
            img = self.transform(img)
        return img, CLASS_TO_IDX[row['label']]


train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.01, 0.05), ratio=(0.3, 3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def make_loaders(soften_exponent=1.0, batch_size=32):
    train_ds = RobotDataset('train.xls', transform=train_transform)
    val_ds = RobotDataset('val.xls', transform=val_transform)
    test_ds = RobotDataset('test.xls', transform=val_transform)

    label_counts = train_ds.df['label'].value_counts()
    total = len(train_ds.df)

    # inverse-frequency weight per sample, softened by `soften_exponent`
    sample_weights = [
        (total / (len(CLASS_NAMES) * label_counts.get(row['label'], 1))) ** soften_exponent
        for _, row in train_ds.df.iterrows()
    ]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    # loss weights always use the full (un-softened) inverse frequency
    class_weights = torch.FloatTensor(
        [total / (len(CLASS_NAMES) * label_counts.get(c, 1)) for c in CLASS_NAMES]
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader, class_weights

## Training and evaluation helpers

`train_one_model` fine-tunes a fresh ResNet-18, saves the best checkpoint by
validation accuracy, and plots the training curve. `evaluate_model` loads a
checkpoint and reports test accuracy, per-class recall, and a confusion
matrix — the numbers behind Table II and Fig. 3 in the report.

In [ ]:
def train_one_model(soften_exponent, model_name, epochs=20, lr=1e-4):
    train_loader, val_loader, test_loader, class_weights = make_loaders(soften_exponent)

    model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    def evaluate(loader):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs).max(1)[1]
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        return correct / total

    best_val_acc = 0.0
    train_losses, val_accs = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        start_time = time.time()

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        scheduler.step()
        val_acc = evaluate(val_loader)
        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)
        val_accs.append(val_acc)

        elapsed = time.time() - start_time
        print(f'Epoch {epoch+1:02d}/{epochs}  loss: {epoch_loss:.4f}  val acc: {val_acc*100:.1f}%  ({elapsed:.0f}s)')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), model_name)
            print(f'  --> best model saved ({val_acc*100:.1f}%)')

    print(f'\nTraining complete. Best val acc: {best_val_acc*100:.1f}%')

    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    epochs_x = list(range(1, len(train_losses) + 1))
    ax1.plot(epochs_x, train_losses, 'b-o', markersize=4, label='Train loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Train loss', color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    ax1.set_xticks(epochs_x)
    ax2 = ax1.twinx()
    ax2.plot(epochs_x, [v * 100 for v in val_accs], 'r-o', markersize=4, label='Val accuracy')
    ax2.set_ylabel('Val accuracy (%)', color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    for x in [6, 11, 16]:
        ax1.axvline(x=x, color='gray', linestyle=':', alpha=0.5)
    plt.title(f'Training curve — {model_name}')
    plt.tight_layout()
    plt.savefig(f'training_curve_{model_name.replace(".pth", "")}.png', dpi=150)
    plt.show()

    return test_loader


def evaluate_model(model_name, test_loader, title):
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    model.load_state_dict(torch.load(model_name, weights_only=True))
    model = model.to(device)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            preds = model(imgs).max(1)[1].cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    print(f'Test accuracy ({title}): {test_acc*100:.1f}%\n')
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.title(f'Confusion matrix — test set ({title})')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{title}.png', dpi=150)
    plt.show()

    return test_acc

## Train v2 — relabelled labels, full inverse-frequency sampling

In [ ]:
test_loader = train_one_model(soften_exponent=1.0, model_name='best_model_v2.pth')
evaluate_model('best_model_v2.pth', test_loader, title='v2')

## Train v3 — softened (w^0.3) sampling

In [ ]:
test_loader = train_one_model(soften_exponent=0.3, model_name='best_model_v3.pth')
evaluate_model('best_model_v3.pth', test_loader, title='v3')

## Result

v3 wins on raw test accuracy (80.5% vs. 77.7%) but v2 has substantially
better recall on the rare `right` class (42% vs. 33%). v2 is the checkpoint
used in the deployment notebooks and the one that actually completed corners
reliably on the track — see the report (Section V) for the full deployment
comparison.